# 02 — Agentes con LangGraph: el RAG como herramienta

Segundo notebook de la serie incremental. En el 01 construimos un pipeline de
RAG multimodal sobre Qdrant; aquí lo convertimos en una **herramienta** que un
**agente** decide cuándo usar.

> **Prerrequisito:** haber ejecutado el notebook 01 (la colección
> `tecnomarket` debe existir en Qdrant) y tener el mismo `.env`.

## ¿Qué es un agente?

Una llamada simple a un LLM es de un solo turno: entra un prompt, sale texto.
Un **agente** envuelve al modelo en un loop **ReAct** (*Reason + Act*, el
patrón del paper [ReAct, 2022](https://arxiv.org/abs/2210.03629)):

```
 ┌─────────────────────────────────────────────────┐
 │  reasoning → action (tool call) → observation   │
 │      ▲                                 │        │
 │      └───────────── repetir ───────────┘        │
 └──────────── hasta que el modelo responde ───────┘
```

Los ingredientes: **tools** (funciones que el modelo puede invocar vía *tool
calling*), **estado** (la conversación acumulada) y una **condición de
parada** (el modelo responde sin pedir tools).

## RAG fijo vs. RAG agéntico

En el notebook 01 el flujo era **fijo**: *siempre* recuperamos top-k y
generamos — aunque la pregunta no lo necesite, y exactamente una vez.

Con el RAG **como tool**, el modelo **decide**:

- *si* buscar (una pregunta de aritmética no necesita el índice),
- *qué* buscar (reformula la consulta en sus propios términos),
- *cuántas veces* buscar (preguntas compuestas → varias búsquedas),
- y puede **combinar** tools (buscar la política de envíos *y* calcular
  un total).

Ese es el salto de "pipeline" a "agente", y LangGraph lo modela como una
**máquina de estados explícita**: **nodes** (unidades de trabajo) y **edges**
(transiciones), con edges *condicionales* para el routing — lo vemos pieza
por pieza en la sección 2.

## Lo que cubrimos

1. El núcleo RAG del notebook 01, compactado.
2. **LangGraph 101** — qué es un node, un edge, el estado, y cómo se
   construye y extiende un `StateGraph`.
3. **Tools** — el RAG como tool, más tools creativas (tracking de pedidos,
   filtro por presupuesto, calculadora).
4. El **grafo del agente** y su loop ReAct.
5. **Playground interactivo** — ver los rounds del agente y el StateGraph
   iluminarse al mismo tiempo.
6. **Reasoning** — activar el *thinking* del modelo y leer su razonamiento.
7. **Fuentes multimodales** — las fotos de producto que el agente recuperó.
8. **Memoria** — checkpointer + `thread_id` para conversaciones multi-turno.
9. **Streamlit app** — todo junto en una app con Docker (`app/`).

In [ ]:
# Carga la configuración desde module4-genai/.env (cópiala de .env.example).
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "rag").exists():          # si ejecutas desde notebooks/
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "768"))
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "https://ollama.com")
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION = os.getenv("QDRANT_COLLECTION", "tecnomarket")

print("Módulo:", ROOT)
print("Embeddings:", GEMINI_EMBEDDING_MODEL, f"({EMBEDDING_DIM} dim)")
print("Qdrant:", QDRANT_URL, "| colección:", COLLECTION)
print("GEMINI_API_KEY definida:", bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")))
print("OLLAMA_API_KEY definida:", bool(os.getenv("OLLAMA_API_KEY")))

## 1. El núcleo de RAG, en versión compacta

Re-empacamos la lógica del notebook 01 en unas pocas funciones: embeddings de
consulta (Gemini), búsqueda densa (Qdrant), BM25 (reconstruido leyendo los
payloads de la colección) y fusión RRF. Nada nuevo — solo más denso, porque
ahora el protagonista es el agente.

También cargamos el **catálogo con sus fotos** (como en el notebook 01):
las usaremos para que el agente muestre *fuentes visuales* (sección 7) y
para las tools estructuradas (sección 3).

In [ ]:
import time
from functools import lru_cache

from google import genai
from google.genai import types
from qdrant_client import QdrantClient, models
from rank_bm25 import BM25Okapi

gclient = genai.Client()
qdrant = QdrantClient(url=QDRANT_URL)

if not qdrant.collection_exists(COLLECTION):
    raise RuntimeError(
        f"La colección '{COLLECTION}' no existe. Ejecuta primero el notebook "
        "01_rag_multimodal.ipynb (sección de ingesta)."
    )

@lru_cache(maxsize=512)
def embed_consulta(texto):
    for intento in range(5):
        try:
            r = gclient.models.embed_content(
                model=GEMINI_EMBEDDING_MODEL,
                contents=f"task: search result | query: {texto}",
                config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
            )
            return list(r.embeddings[0].values)
        except Exception:
            time.sleep(2 ** intento)
    raise RuntimeError("API de embeddings sin respuesta.")

# Reconstruimos el corpus léxico desde los payloads de Qdrant (en el notebook
# 01 lo teníamos en memoria; aquí la colección es la fuente de la verdad).
_puntos, _ = qdrant.scroll(collection_name=COLLECTION, limit=1000, with_payload=True)
PAYLOAD_POR_REF = {p.payload["ref"]: p.payload for p in _puntos}
_corpus = [(ref, pl["texto"]) for ref, pl in PAYLOAD_POR_REF.items() if pl["texto"]]
_refs_bm25 = [ref for ref, _ in _corpus]
_bm25 = BM25Okapi([t.lower().split() for _, t in _corpus])

def buscar_hibrido(consulta, top_k=4, pool=15, rrf_k=60):
    densos = qdrant.query_points(collection_name=COLLECTION,
                                 query=embed_consulta(consulta),
                                 limit=pool, with_payload=True).points
    scores = _bm25.get_scores(consulta.lower().split())
    orden_lex = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    fusion = {}
    for rank, h in enumerate(densos):
        fusion[h.payload["ref"]] = fusion.get(h.payload["ref"], 0) + 1 / (rrf_k + rank)
    for rank, i in enumerate(orden_lex[:pool]):
        if scores[i] > 0:
            ref = _refs_bm25[i]
            fusion[ref] = fusion.get(ref, 0) + 1 / (rrf_k + rank)
    mejores = sorted(fusion, key=fusion.get, reverse=True)[:top_k]
    return [PAYLOAD_POR_REF[ref] for ref in mejores]

print("Núcleo RAG listo. Puntos en la colección:", len(PAYLOAD_POR_REF))

In [ ]:
# Catálogo + helpers de imágenes (mismos trucos del notebook 01).
import base64
import json

CATALOG_DIR = ROOT / "rag" / "catalog"
PRODUCTS = json.loads((CATALOG_DIR / "products.json").read_text(encoding="utf-8"))
PROD_POR_SKU = {p["sku"]: p for p in PRODUCTS}

from IPython.display import HTML, display

def img_b64(path):
    return "data:image/jpeg;base64," + base64.b64encode(Path(path).read_bytes()).decode()

def galeria(productos, ancho=120):
    tarjetas = []
    for p in productos:
        src = img_b64(CATALOG_DIR / "images" / p["imagen"])
        tarjetas.append(
            f'<div style="display:inline-block;margin:6px;text-align:center;width:{ancho}px;'
            f'vertical-align:top;font-family:sans-serif;font-size:11px">'
            f'<img src="{src}" style="width:{ancho}px;height:{ancho}px;object-fit:cover;'
            f'border-radius:8px"><br><b>{p["sku"]}</b><br>{p["nombre"]}<br>'
            f'${p["precio"]:,.0f}</div>'
        )
    return HTML("<div>" + "".join(tarjetas) + "</div>")

print(f"{len(PRODUCTS)} productos cargados con foto y precio.")
display(galeria(PRODUCTS[:4]))

## 2. LangGraph 101: estado, nodes y edges

LangGraph modela un agente como un **grafo de estados** (`StateGraph`). Antes
de meter un LLM, entendamos las tres piezas con un ejemplo de juguete:

### El estado (State)

El estado es **un diccionario compartido** que viaja por el grafo: cada node
lo lee y devuelve los campos que quiere cambiar. Se declara como `TypedDict`
(un dict con campos y tipos fijos).

### ¿Qué es un reducer?

Cuando un node devuelve un campo que **ya tenía valor**, hay que decidir qué
hacer con los dos valores (el viejo y el nuevo). Esa decisión es el
**reducer**: una función `(viejo, nuevo) → resultado`.

Solo hay dos casos que nos importan:

| Campo declarado como… | El node devuelve `x` | Resultado |
|---|---|---|
| `x: str` (sin reducer) | `"hola"` | `"hola"` — **reemplaza** lo que había |
| `x: Annotated[list, operator.add]` | `["hola"]` | `viejo + ["hola"]` — **se suma a la lista** |

Es decir: sin reducer, el último node "gana"; con reducer, los aportes de
todos los nodes **se acumulan**. Para un chat queremos lo segundo — que cada
mensaje se agregue al historial, no que lo borre — y para eso existe
`add_messages`, el reducer de LangGraph para listas de mensajes
(esencialmente `viejo + nuevo`, con detalles extra como deduplicar por id).

La sintaxis `Annotated[list, operator.add]` se lee: *"este campo es una
`list`, y su reducer es `operator.add`"* — `Annotated` solo adjunta ese dato
extra al tipo.

### El node

Una **función de Python** que recibe el estado y devuelve un **update
parcial** (un dict con solo los campos que cambia). Eso es todo — un node no
sabe nada del resto del grafo. Se registra con `add_node("nombre", funcion)`.

### El edge

La **transición** entre nodes. Dos tipos:

- **Edge fijo** — `add_edge("a", "b")`: después de `a` SIEMPRE viene `b`.
- **Edge condicional** — `add_conditional_edges("a", router, mapa)`: después
  de `a` se ejecuta la función `router(estado)`, que devuelve una etiqueta, y
  el `mapa` traduce esa etiqueta al node destino. Así se implementa el
  *branching* (y el loop del agente).

Dos nodes especiales: `START` (por dónde entra el input) y `END` (terminar).

### La receta para construir (o extender) un grafo

```python
grafo = StateGraph(MiEstado)        # 1. declarar el estado
grafo.add_node("a", nodo_a)         # 2. registrar nodes
grafo.add_node("b", nodo_b)         #    (extender = añadir más add_node
grafo.add_edge(START, "a")          #     y cablearlos con edges ANTES
grafo.add_conditional_edges(...)    #     de compilar)
grafo.add_edge("b", END)
app = grafo.compile()               # 3. compilar → objeto ejecutable
```

⚠ **`compile()` congela el grafo**: el objeto compilado es inmutable. Para
extenderlo (otro node, otra tool) se agrega al *builder* y se recompila —
por eso conviene encapsular la construcción en una función.

In [ ]:
# Grafo de juguete SIN LLM, para ver la mecánica pura.
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

class EstadoDemo(TypedDict):
    texto: str                              # sin reducer → se sobrescribe
    trazas: Annotated[list[str], operator.add]  # con reducer → se acumula

# --- Nodes: funciones estado → update parcial ---
def normalizar(estado: EstadoDemo) -> dict:
    return {"texto": estado["texto"].strip().lower(),
            "trazas": ["normalizar: limpié el texto"]}

def responder_pregunta(estado: EstadoDemo) -> dict:
    return {"texto": f"Buena pregunta: '{estado['texto']}'",
            "trazas": ["responder_pregunta"]}

def hacer_eco(estado: EstadoDemo) -> dict:
    return {"texto": estado["texto"].upper() + "!!!",
            "trazas": ["hacer_eco"]}

# --- Router del edge condicional: devuelve una ETIQUETA, no un node ---
def es_pregunta(estado: EstadoDemo) -> str:
    return "pregunta" if estado["texto"].endswith("?") else "afirmacion"

# --- Construcción ---
demo = StateGraph(EstadoDemo)
demo.add_node("normalizar", normalizar)
demo.add_node("responder_pregunta", responder_pregunta)
demo.add_node("hacer_eco", hacer_eco)
demo.add_edge(START, "normalizar")
demo.add_conditional_edges("normalizar", es_pregunta,
                           {"pregunta": "responder_pregunta",
                            "afirmacion": "hacer_eco"})
demo.add_edge("responder_pregunta", END)
demo.add_edge("hacer_eco", END)
app_demo = demo.compile()

for entrada in ["  ¿Hay envío gratis?  ", "me gusta esta tienda"]:
    final = app_demo.invoke({"texto": entrada, "trazas": []})
    print(f"{entrada!r:32} → {final['texto']!r}")
    print(f"{'':32}   trazas acumuladas: {final['trazas']}")

In [ ]:
# El grafo se dibuja solo (Mermaid). Compara el diagrama con el código:
# cajas = nodes, flechas sólidas = edges fijos, punteadas = condicionales.
from IPython.display import Image, display

try:
    display(Image(app_demo.get_graph().draw_mermaid_png()))
except Exception:
    print(app_demo.get_graph().draw_ascii())

In [ ]:
# EXTENDER un grafo = volver al builder, añadir el node, recablear, recompilar.
# Insertamos un node "censurar" entre los dos finales y END.
#
# ⚠ LangGraph avisará "Adding a node to a graph that has already been
# compiled…" — exactamente el punto: app_demo (ya compilado) NO cambia;
# los cambios solo existen en el builder hasta el nuevo compile().
def censurar(estado: EstadoDemo) -> dict:
    return {"texto": estado["texto"].replace("!!!", "."),
            "trazas": ["censurar: bajé el tono"]}

demo.add_node("censurar", censurar)
demo.edges.discard(("responder_pregunta", END))   # quitamos los edges viejos
demo.edges.discard(("hacer_eco", END))
demo.add_edge("responder_pregunta", "censurar")
demo.add_edge("hacer_eco", "censurar")
demo.add_edge("censurar", END)
app_demo_v2 = demo.compile()                      # nueva versión ejecutable

print(app_demo_v2.invoke({"texto": "me gusta esta tienda", "trazas": []})["texto"])
try:
    display(Image(app_demo_v2.get_graph().draw_mermaid_png()))
except Exception:
    print(app_demo_v2.get_graph().draw_ascii())

Con esa mecánica clara, el agente es solo un caso particular: un node que
llama al LLM, un node que ejecuta tools, y un edge condicional que pregunta
*"¿el modelo pidió tools?"* para cerrar el loop ReAct.

## 3. Las tools

Cinco tools declaradas con `@tool` de LangChain. El **docstring y los type
hints se convierten en el JSON Schema** que el modelo ve — descríbelos
pensando en el modelo, no en humanos: de esa descripción depende que el agente
sepa *cuándo* invocar cada una.

| Tool | Qué enseña |
|---|---|
| `buscar_tecnomarket` | el RAG del notebook 01, ahora como tool (búsqueda semántica) |
| `detalle_producto` | lookup exacto por SKU (filtro de payload, sin embeddings) |
| `productos_por_presupuesto` | filtro **estructurado** (categoría + precio): no todo es semántico |
| `estado_pedido` | envolver un "backend" (aquí simulado): tracking de pedidos |
| `calculadora` | combinar conocimiento recuperado con cálculo |

In [ ]:
import hashlib
from datetime import date, timedelta

from langchain_core.tools import tool

# Registro de fuentes: cada búsqueda anota los SKUs que recuperó, para poder
# mostrar las FOTOS de esos productos después de la respuesta (sección 7).
FUENTES_RECIENTES: list[dict] = []

@tool
def buscar_tecnomarket(consulta: str) -> str:
    'Busca en la base de conocimiento de TecnoMarket (políticas de envíos, devoluciones y garantías, y catálogo de productos). Recibe una consulta en lenguaje natural y devuelve los pasajes más relevantes.'
    resultados = buscar_hibrido(consulta, top_k=4)
    FUENTES_RECIENTES.extend(r for r in resultados if r.get("sku"))
    bloques = []
    for i, r in enumerate(resultados, 1):
        if r["tipo"] == "producto_imagen":
            bloques.append(f"[{i}] foto del producto {r['sku']} — {r['nombre']}")
        else:
            origen = r.get("fuente") or f"{r['sku']} — {r['nombre']}"
            bloques.append(f"[{i}] (fuente: {origen})\n{r['texto']}")
    return "\n\n".join(bloques) if bloques else "Sin resultados."

@tool
def detalle_producto(sku: str) -> str:
    'Devuelve la ficha exacta de un producto de TecnoMarket dado su SKU (formato TM-XXXX): nombre, categoría, precio y descripción.'
    p = PROD_POR_SKU.get(sku.strip().upper())
    if not p:
        return f"No existe el SKU {sku}."
    FUENTES_RECIENTES.append({"sku": p["sku"], "nombre": p["nombre"],
                              "imagen": p["imagen"]})
    return json.dumps({k: p[k] for k in ("sku", "nombre", "categoria",
                                         "precio", "descripcion")},
                      ensure_ascii=False)

@tool
def productos_por_presupuesto(categoria: str, presupuesto: float) -> str:
    'Lista los productos de TecnoMarket de una categoría (p. ej. "audifonos", "zapatillas", "relojes", "cafeteras") con precio menor o igual al presupuesto dado, ordenados del más barato al más caro.'
    cat = categoria.strip().lower()
    encontrados = sorted(
        (p for p in PRODUCTS
         if p["categoria"] == cat and p["precio"] <= presupuesto),
        key=lambda p: p["precio"])[:5]
    if not encontrados:
        categorias = sorted({p["categoria"] for p in PRODUCTS})
        return (f"Nada en '{categoria}' por ≤ {presupuesto}. "
                f"Categorías disponibles: {', '.join(categorias)}.")
    FUENTES_RECIENTES.extend({"sku": p["sku"], "nombre": p["nombre"],
                              "imagen": p["imagen"]} for p in encontrados)
    return json.dumps([{k: p[k] for k in ("sku", "nombre", "precio")}
                       for p in encontrados], ensure_ascii=False)

@tool
def estado_pedido(numero_pedido: str) -> str:
    'Consulta el estado de un pedido de TecnoMarket dado su número (formato PED-XXXX): estado actual, transportadora y fecha estimada de entrega.'
    # Simulación determinista: el hash del número decide el estado. En
    # producción esta tool llamaría al API interno de órdenes — al agente
    # le da igual: solo ve el docstring y el resultado.
    n = int(hashlib.sha1(numero_pedido.strip().upper().encode()).hexdigest(), 16)
    estados = ["confirmado, preparando el paquete", "despachado, en tránsito",
               "en el centro de distribución local", "en reparto, llega hoy"]
    entrega = date.today() + timedelta(days=n % 4 + 1)
    return json.dumps({"pedido": numero_pedido.strip().upper(),
                       "estado": estados[n % len(estados)],
                       "transportadora": ["VelozExpress", "AndesCargo"][n % 2],
                       "entrega_estimada": entrega.isoformat()},
                      ensure_ascii=False)

@tool
def calculadora(expresion: str) -> str:
    'Evalúa una expresión aritmética simple, p. ej. "2 * 129.0 + 12000".'
    if not set(expresion) <= set("0123456789+-*/(). "):
        return "Error: solo se admite aritmética básica."
    try:
        return str(eval(expresion, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: {exc}"

TOOLS = [buscar_tecnomarket, detalle_producto, productos_por_presupuesto,
         estado_pedido, calculadora]
print("Tools:", [t.name for t in TOOLS])
print("\nPruebas directas (sin LLM):")
print(" ", detalle_producto.invoke({"sku": "TM-1008"})[:100], "…")
print(" ", productos_por_presupuesto.invoke(
    {"categoria": "audifonos", "presupuesto": 200}))
print(" ", estado_pedido.invoke({"numero_pedido": "PED-7431"}))

## 4. El LLM (Ollama cloud) con las tools enlazadas

Misma celda de conexión del notebook 01 (elige el modelo en `OLLAMA_MODEL`;
para agentes conviene uno con buen *tool calling*). `bind_tools` adjunta los
JSON Schemas de las tools a cada llamada, habilitando el *tool calling*
nativo del modelo.

In [ ]:
# ⚙️ El modelo de Ollama cloud se elige AQUÍ, en el notebook.
#    Catálogo: https://ollama.com/search?c=cloud . Algunas opciones:
#      "minimax-m3:cloud"    razonador (thinking)
#      "kimi-k3:cloud"       multimodal (visión); se factura como "extra usage"
#      "gpt-oss:120b-cloud"  incluido en el plan gratuito
OLLAMA_MODEL = "minimax-m3:cloud"

# Si el modelo elegido no está disponible en tu plan (p. ej. HTTP 402 por saldo
# de extra usage en cero), caemos automáticamente al fallback del plan gratuito.
OLLAMA_FALLBACK = "gpt-oss:120b-cloud"

from langchain_ollama import ChatOllama
from ollama import Client as OllamaClient

OLLAMA_HEADERS = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}

def conectar_llm(**kwargs):
    # kwargs extra van directo a ChatOllama (p. ej. reasoning=True).
    probe = OllamaClient(host=OLLAMA_HOST, headers=OLLAMA_HEADERS)
    for modelo in [OLLAMA_MODEL, OLLAMA_FALLBACK]:
        try:
            probe.chat(model=modelo,
                       messages=[{"role": "user", "content": "ok"}],
                       options={"num_predict": 1})
        except Exception as exc:
            print(f"⚠ {modelo} no disponible: {str(exc)[:110]}")
            continue
        print("✔ Usando el modelo:", modelo)
        return ChatOllama(
            model=modelo,
            base_url=OLLAMA_HOST,
            client_kwargs={"headers": OLLAMA_HEADERS},
            temperature=0.1,
            **kwargs,
        )
    raise RuntimeError("Ningún modelo de Ollama cloud respondió; revisa OLLAMA_API_KEY.")

llm = conectar_llm()

## 5. El grafo del agente

Con el vocabulario de la sección 2, el agente es un `StateGraph` de **dos
nodes y un loop**:

- **Estado**: la lista `messages`, con el reducer `add_messages` (cada node
  *agrega* mensajes en lugar de sobrescribir la lista).
- **Node `agente`**: llama al LLM (con las tools enlazadas), anteponiendo el
  **system prompt** — en un agente no hay system prompt implícito: el modelo
  ve exactamente los mensajes que le pasamos, así que lo inyectamos nosotros
  en cada llamada.
- **Node `tools`**: `ToolNode` (prefabricado de LangGraph) ejecuta las tools
  que el modelo pidió y agrega sus resultados como `ToolMessage`s.
- **Edge condicional**: `tools_condition` rutea — ¿el último mensaje pide
  tools? → `tools`; ¿no? → `END`.
- **Edge fijo `tools → agente`**: cierra el loop ReAct.

```
START → agente ─(¿pidió tools?)─ sí → tools ─┐
          ▲               │                  │
          └───────────────┼──────────────────┘
                          no → END
```

In [ ]:
from langchain_core.messages import (AIMessage, HumanMessage, SystemMessage,
                                     ToolMessage)
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class EstadoAgente(TypedDict):
    messages: Annotated[list, add_messages]

# ¿Y el system prompt? En un agente NO aparece solo: el modelo recibe
# exactamente estado["messages"]. Lo definimos aquí y el node agente lo
# antepone EN CADA LLAMADA (sin guardarlo en el estado: así no se duplica
# en la memoria y se puede cambiar sin tocar los checkpoints).
# Ojo: las descripciones de las tools NO van aquí — bind_tools las envía
# aparte, en el campo `tools` del API (inspecciónalas con
# llm.bind_tools(TOOLS).kwargs["tools"]).
SYSTEM_PROMPT = (
    "Eres el asistente de la tienda TecnoMarket. Responde en español, breve "
    "y amable. Usa las tools para consultar catálogo, políticas y pedidos: "
    "no inventes SKUs, precios ni plazos. Si no encuentras la información, "
    "dilo claramente.")

def construir_agente(llm_base, checkpointer=None):
    # Encapsular la construcción (sección 2) permite recompilar variantes:
    # otro LLM (reasoning, sección 6) u otra memoria (sección 8).
    llm_con_tools = llm_base.bind_tools(TOOLS)

    def nodo_agente(estado: EstadoAgente) -> dict:
        mensajes = [SystemMessage(content=SYSTEM_PROMPT)] + estado["messages"]
        return {"messages": [llm_con_tools.invoke(mensajes)]}

    grafo = StateGraph(EstadoAgente)
    grafo.add_node("agente", nodo_agente)
    grafo.add_node("tools", ToolNode(TOOLS))
    grafo.add_edge(START, "agente")
    grafo.add_conditional_edges("agente", tools_condition,
                                {"tools": "tools", END: END})
    grafo.add_edge("tools", "agente")
    return grafo.compile(checkpointer=checkpointer)

agente = construir_agente(llm)
print("Grafo compilado.")

try:
    display(Image(agente.get_graph().draw_mermaid_png()))
except Exception:
    print(agente.get_graph().draw_ascii())

## 6. Playground: los rounds del agente y el StateGraph, en vivo

Cada iteración del loop ReAct es un **round**: se ejecuta un node y el estado
crece. Con `stream_mode="updates"` el grafo emite `{node: update}` tras cada
node — sabemos exactamente **dónde está** la ejecución.

El panel de abajo junta las dos vistas:

- **Izquierda** — el StateGraph con el node activo **iluminado**.
- **Derecha** — el transcript: tool calls (con argumentos), resultados de las
  tools y la respuesta final.

Y se puede avanzar **round a round** (botón *⏭ Paso*) — ideal para narrar en
clase qué decidió el modelo en cada transición — o dejarlo correr (*▶ Hasta
el final*).

Preguntas sugeridas (cada una ejercita tools distintas):

- *¿Cuánto costarían dos teclados TM-1008 con el envío estándar incluido?* →
  ficha + búsqueda + calculadora.
- *¿Puedo devolver unos audífonos abiertos? ¿y cuántos meses de garantía
  tienen?* → dos búsquedas reformuladas (RAG agéntico).
- *¿Dónde está mi pedido PED-7431?* → tracking simulado.
- *Recomiéndame audífonos por menos de 200* → filtro estructurado.
- *¿Cuánto es 15% de 549?* → sin tools de búsqueda: el índice no hace falta.

In [ ]:
import html as html_mod

import ipywidgets as W

NODOS_VISTA = ["START", "agente", "tools", "END"]

def render_grafo(activo=None):
    # Dibujo HTML del StateGraph con el node activo resaltado. (Para grafos
    # grandes usaríamos draw_mermaid_png; aquí lo pintamos a mano para poder
    # iluminarlo en cada round.)
    def caja(n):
        on = n == activo
        estilo = ("background:#ff6f00;color:white;border:2px solid #e65100"
                  if on else "background:#f5f5f5;color:#333;border:1px solid #bbb")
        forma = "border-radius:16px" if n in ("START", "END") else "border-radius:6px"
        return (f'<div style="{estilo};{forma};padding:6px 18px;margin:4px auto;'
                f'width:90px;text-align:center;font-family:monospace;'
                f'font-size:13px">{n}</div>')
    flecha = '<div style="text-align:center;color:#888">↓</div>'
    loop = ('<div style="text-align:center;color:#888;font-size:11px">'
            '⇄ (¿pidió tools? sí → tools → agente; no → END)</div>')
    return (f'<div style="width:270px">{caja("START")}{flecha}{caja("agente")}'
            f'{loop}{caja("tools")}{flecha}{caja("END")}</div>')

def describir(msg):
    # Un mensaje del estado → una línea legible del transcript.
    if isinstance(msg, AIMessage):
        piezas = []
        razonamiento = msg.additional_kwargs.get("reasoning_content")
        if razonamiento:
            piezas.append(f"🧠 <i>{html_mod.escape(razonamiento[:300])}…</i>")
        for tc in msg.tool_calls:
            piezas.append(f"🛠 <b>tool call</b> → <code>{tc['name']}"
                          f"({json.dumps(tc['args'], ensure_ascii=False)})</code>")
        if msg.content:
            piezas.append(f"💬 <b>respuesta:</b> {html_mod.escape(msg.content)}")
        return "<br>".join(piezas)
    if isinstance(msg, ToolMessage):
        return (f"📦 <code>{msg.name}</code> devolvió: "
                f"<span style='color:#666'>{html_mod.escape(str(msg.content)[:280])}…</span>")
    return html_mod.escape(str(msg.content))

_pg_q = W.Text(value="¿Cuánto costarían dos teclados TM-1008 con el envío estándar incluido?",
               layout=W.Layout(width="640px"), description="Pregunta:")
_pg_nueva = W.Button(description="🔄 Nueva pregunta", button_style="primary")
_pg_paso = W.Button(description="⏭ Paso", disabled=True)
_pg_todo = W.Button(description="▶ Hasta el final", disabled=True)
_pg_grafo = W.HTML(value=render_grafo())
_pg_log = W.Output(layout=W.Layout(border="1px solid #ddd", width="640px",
                                   height="380px", overflow="auto"))
_pg_iter = None
_pg_round = 0

def _log_html(texto):
    with _pg_log:
        display(HTML(f'<div style="font-family:sans-serif;font-size:12px;'
                     f'margin:4px 6px">{texto}</div>'))

def _pg_reset(_):
    global _pg_iter, _pg_round
    _pg_log.clear_output()
    _pg_round = 0
    _pg_iter = agente.stream(
        {"messages": [HumanMessage(content=_pg_q.value)]}, stream_mode="updates")
    _pg_grafo.value = render_grafo("START")
    _log_html(f"👤 <b>{html_mod.escape(_pg_q.value)}</b>")
    _pg_paso.disabled = _pg_todo.disabled = False

def _pg_avanzar(_):
    global _pg_iter, _pg_round
    item = next(_pg_iter, None)
    if item is None:
        _pg_grafo.value = render_grafo("END")
        _log_html("🏁 <b>END</b> — el modelo respondió sin pedir más tools.")
        _pg_paso.disabled = _pg_todo.disabled = True
        return False
    nodo, update = next(iter(item.items()))
    _pg_round += 1
    _pg_grafo.value = render_grafo(nodo)
    _log_html(f'<span style="color:#ff6f00"><b>— round {_pg_round}: '
              f'node <code>{nodo}</code> —</b></span>')
    for msg in update["messages"]:
        _log_html(describir(msg))
    return True

def _pg_correr(_):
    while _pg_avanzar(None):
        pass

_pg_nueva.on_click(_pg_reset)
_pg_paso.on_click(_pg_avanzar)
_pg_todo.on_click(_pg_correr)
display(W.VBox([_pg_q, W.HBox([_pg_nueva, _pg_paso, _pg_todo]),
                W.HBox([_pg_grafo, _pg_log])]))
_pg_reset(None)  # deja la primera pregunta lista: pulsa ⏭ Paso round a round

Fíjate en el segundo ejemplo (la pregunta compuesta de devoluciones +
garantías): el agente suele lanzar **dos tool calls con consultas
reformuladas** en lugar de una sola mezclada — eso es **RAG agéntico**, lo
que el pipeline fijo del notebook 01 no podía hacer.

## 7. Reasoning: ver al modelo *pensar*

`minimax-m3` es un modelo **razonador** (*thinking model*): antes de la
respuesta genera *tokens de razonamiento* — un borrador interno donde planea,
se corrige y decide. Por defecto `ChatOllama` no lo pide; se activa con
`reasoning=True`, y el texto llega **separado** de la respuesta, en
`additional_kwargs["reasoning_content"]`.

¿Por qué importa en agentes? El razonamiento muestra **por qué eligió una
tool** ("el usuario pregunta por un precio → necesito la ficha…"), que es
justo lo que queremos auditar (y enseñar). El costo: más tokens y más
latencia — en producción se activa selectivamente.

> Si elegiste un modelo sin thinking (p. ej. `gpt-oss` responde razonamiento
> vacío o no soporta el flag), la celda lo detecta y lo dice.

In [ ]:
# El mismo conectar_llm de siempre, ahora con reasoning=True.
llm_razonador = conectar_llm(reasoning=True)

resp = llm_razonador.invoke(
    "Tengo 3 pedidos de 2 teclados cada uno a 129 USD el teclado, con 10% de "
    "descuento sobre el total. ¿Cuánto pago? Responde en una frase.")

razonamiento = resp.additional_kwargs.get("reasoning_content", "")
if razonamiento:
    print("=== REASONING (borrador interno del modelo) ===")
    print(razonamiento[:1200])
    print("\n=== RESPUESTA (lo que vería el usuario) ===")
else:
    print(f"(El modelo '{llm_razonador.model}' no expone reasoning; "
          "prueba con minimax-m3:cloud.)\n")
print(resp.content)

In [ ]:
# Reasoning DENTRO del agente: recompilamos el grafo con el LLM razonador
# (por eso construir_agente recibe el LLM como parámetro). El playground de
# la sección 6 ya muestra el 🧠 cuando un AIMessage trae reasoning_content:
# re-apunta la variable y vuelve a usarlo si quieres verlo con widgets.
agente_razonador = construir_agente(llm_razonador)

for paso in agente_razonador.stream(
        {"messages": [HumanMessage(content=
            "¿Me conviene más el pedido PED-7431 o comprar de nuevo unos "
            "audífonos de menos de 150 USD? Revisa el estado del pedido y "
            "las opciones antes de responder.")]},
        stream_mode="updates"):
    nodo, update = next(iter(paso.items()))
    print(f"\n════ node: {nodo} ════")
    for msg in update["messages"]:
        r = getattr(msg, "additional_kwargs", {}).get("reasoning_content")
        if r:
            print(f"🧠 reasoning: {r[:400]}…\n")
        msg.pretty_print()

## 8. Fuentes multimodales: mostrar lo que el agente recuperó

Como en el notebook 01, no basta con leer la respuesta: hay que **auditar las
fuentes**. Las tools registran los productos que tocaron
(`FUENTES_RECIENTES`), así que después de cada corrida podemos renderizar la
**galería de fotos** de los productos que respaldan la respuesta — RAG
multimodal, ahora dentro del agente.

In [ ]:
def preguntar_con_fuentes(pregunta, agente_a_usar=None):
    ag = agente_a_usar or agente
    FUENTES_RECIENTES.clear()
    final = ag.invoke({"messages": [HumanMessage(content=pregunta)]})
    print(final["messages"][-1].content)
    vistos, productos = set(), []
    for f in FUENTES_RECIENTES:
        p = PROD_POR_SKU.get(f["sku"])
        if p and p["sku"] not in vistos:
            vistos.add(p["sku"])
            productos.append(p)
    if productos:
        display(HTML("<hr><b>🖼 Productos consultados por el agente:</b>"))
        display(galeria(productos[:6]))

preguntar_con_fuentes(
    "Busco unas zapatillas de lona para uso diario y quiero saber si podría "
    "devolverlas en caso de que no me queden bien.")

## 9. Memoria: checkpointer + `thread_id`

Hasta ahora cada `invoke` **empieza de cero**: el estado vive solo durante la
corrida. Pregunta "¿cuánto cuesta el TM-1008?" y luego "¿y su garantía?" — el
agente no sabe de qué producto hablas.

LangGraph resuelve esto con un **checkpointer**. Desarmemos la pieza:

### ¿QUÉ se guarda exactamente?

**El estado completo del grafo** — en nuestro caso, la lista `messages`
entera. Y ojo: eso no es solo "pregunta y respuesta"; es **todo el tráfico
del loop ReAct**:

| # | Tipo de mensaje | Contenido |
|---|---|---|
| 1 | `HumanMessage` | "¿Cuánto cuesta el teclado TM-1008?" |
| 2 | `AIMessage` | tool call → `detalle_producto({"sku": "TM-1008"})` |
| 3 | `ToolMessage` | el JSON que devolvió la tool (¡con el precio!) |
| 4 | `AIMessage` | la respuesta final al usuario |

Los 4 mensajes quedan guardados. Por eso el turno 2 puede resolver "¿y su
garantía?": el modelo **re-lee** ese historial (incluido el `ToolMessage` con
la ficha del TM-1008) antes de responder.

### ¿CUÁNDO y DÓNDE se guarda?

- **Cuándo**: después de **cada node** (no al final de la corrida), el
  checkpointer persiste una foto del estado — un **checkpoint**. Una corrida
  con 2 tool calls deja ~5 checkpoints. Bonus: si el proceso muere a mitad
  del loop, se puede reanudar desde el último checkpoint.
- **Dónde**: en el backend del checkpointer, indexado por **`thread_id`** (el
  identificador de la conversación) + un id de checkpoint. `MemorySaver` = un
  diccionario en RAM (se pierde al reiniciar el kernel — perfecto para
  demos). En producción, mismo API con backend persistente (`SqliteSaver`,
  `PostgresSaver`): solo cambia el objeto que pasas a `compile()`.

### ¿CÓMO funciona el turno 2, mecánicamente?

```
invoke(mensaje_nuevo, thread_id="cliente-ana")
  1. el checkpointer CARGA el último checkpoint del thread "cliente-ana"
  2. add_messages (el reducer del estado) AGREGA el mensaje nuevo al final
  3. el node agente pasa TODA la lista al LLM → contexto completo
  4. cada node nuevo vuelve a guardar checkpoint
```

Es decir: la "memoria" no es magia ni un resumen — es **re-enviar el
historial completo** en cada turno. Consecuencia práctica: el contexto (y el
costo por token) **crece con la conversación**; en producción se recorta o
resume el historial pasado cierto tamaño.

- Cada `thread_id` es un universo aparte: la memoria de un cliente no se
  filtra a otro (lo probamos abajo).
- Esto es memoria de **corto plazo** (la conversación). La memoria de *largo
  plazo* (preferencias del usuario entre conversaciones) es otra pieza: un
  *store* consultable — buena extensión para explorar.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memoria = MemorySaver()
agente_con_memoria = construir_agente(llm, checkpointer=memoria)

hilo_ana = {"configurable": {"thread_id": "cliente-ana"}}

# Turno 1: preguntamos por un producto concreto.
r1 = agente_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿Cuánto cuesta el teclado TM-1008?")]},
    config=hilo_ana)
print("— Turno 1:", r1["messages"][-1].content[:200], "\n")

# Turno 2: "¿y garantía?" — SIN repetir el SKU. El checkpoint lo recuerda.
r2 = agente_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿Y cuántos meses de garantía tiene?")]},
    config=hilo_ana)
print("— Turno 2:", r2["messages"][-1].content[:300])

In [ ]:
# La prueba de aislamiento: otro thread_id = otra conversación = amnesia.
hilo_luis = {"configurable": {"thread_id": "cliente-luis"}}
r3 = agente_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿Y cuántos meses de garantía tiene?")]},
    config=hilo_luis)
print("— Mismo mensaje, thread nuevo:", r3["messages"][-1].content[:300])

In [ ]:
# Autopsia de la memoria: ¿qué hay EXACTAMENTE guardado en el thread de Ana?
# get_state(config) devuelve el último checkpoint; .values es nuestro estado.
estado_ana = agente_con_memoria.get_state(hilo_ana)

print(f"Mensajes guardados en 'cliente-ana': "
      f"{len(estado_ana.values['messages'])}\n")
for i, m in enumerate(estado_ana.values["messages"], 1):
    tipo = type(m).__name__
    if isinstance(m, AIMessage) and m.tool_calls:
        detalle = "tool call → " + ", ".join(
            f"{tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})"
            for tc in m.tool_calls)
    else:
        detalle = str(m.content).replace("\n", " ")[:90]
    print(f"  {i:>2}. {tipo:<14} {detalle}")

# Nota cómo los ToolMessage (las fichas, los pasajes del RAG) TAMBIÉN están
# en la memoria: por eso el turno 2 sabía el precio sin volver a buscar.

# Y el historial de checkpoints: una foto por node ejecutado en el thread.
n_checkpoints = sum(1 for _ in agente_con_memoria.get_state_history(hilo_ana))
print(f"\nCheckpoints acumulados en el thread: {n_checkpoints} "
      "(uno por node; sirven para reanudar o hacer time-travel)")

print("Mensajes en el hilo de Luis:",
      len(agente_con_memoria.get_state(hilo_luis).values["messages"]),
      "(su universo aparte)")

## 10. Todo junto: la app de Streamlit (con Docker)

Todo lo del notebook — agente + tools + reasoning + fuentes con foto +
memoria por thread — está empaquetado en una **app de Streamlit** en
`app/streamlit_app.py`, lista para correr en Docker:

```bash
# 1. Qdrant arriba y colección ingestada (notebook 01)
cd qdrant && docker compose up -d && cd ..

# 2. Construir y lanzar la app (lee las keys de tu .env)
cd app && docker compose up --build
```

→ abre <http://localhost:8501>

La app muestra en la barra lateral el **modelo** (elegible), el **toggle de
reasoning**, el **StateGraph** y el botón de *nueva conversación* (que
estrena `thread_id` — la memoria de la sección 9). Cada respuesta despliega
los **rounds** del loop ReAct (tool calls y resultados), el razonamiento del
modelo y las **fotos** de los productos consultados.

> El contenedor alcanza al Qdrant del host vía `host.docker.internal:6333`;
> las API keys entran por `env_file: ../.env`. Sin Docker también corre:
> `uv run streamlit run app/streamlit_app.py`.

## Resumen

- **LangGraph 101**: estado tipado con reducers, nodes = funciones que
  devuelven updates parciales, edges fijos y condicionales; `compile()`
  congela — extender = añadir al builder y recompilar.
- Un **agente** = LLM + tools + estado + loop ReAct: `agente ⇄ tools` con
  routing de `tools_condition`.
- El RAG del notebook 01 se volvió la tool `buscar_tecnomarket`, conviviendo
  con tools estructuradas (`detalle_producto`, `productos_por_presupuesto`),
  de "backend" (`estado_pedido`) y de cálculo (`calculadora`). La calidad de
  los **docstrings** es parte del prompt.
- `stream_mode="updates"` emite `{node: update}` por round — con eso armamos
  el playground que ilumina el StateGraph en vivo.
- **Reasoning** (`reasoning=True`) separa el borrador interno del modelo de
  su respuesta: auditable en `additional_kwargs["reasoning_content"]`.
- **Memoria** = checkpointer + `thread_id`: estado persistente por
  conversación sin cambiar el grafo.
- Todo junto corre como app en `app/` (Streamlit + Docker).

**Siguiente** → `03_mcp_rag_agentes.ipynb`: sacamos estas tools del notebook
y las servimos por **MCP**, el protocolo estándar para conectar tools a
*cualquier* host de LLM.